# Multimodal RAG with LlamaIndex and NeMo Retriever

This notebook shows how to ingest a multimodal PDF with the current `nemo_retriever` library API, store the extracted text, table, chart, and image content in local LanceDB, and query it with a LlamaIndex RAG pipeline.

**Note:** This example runs NeMo Retriever locally in-process and writes to a LanceDB directory on disk. You need Python 3.12, the NeMo Retriever Library installed with the local model dependencies, the sample `data/multimodal_test.pdf`, and `NVIDIA_API_KEY` for the hosted NVIDIA LLM used to synthesize the final answer.

Install the framework packages for this notebook and NeMo Retriever from this source checkout.

In [ ]:
%pip install -qU llama-index llama-index-llms-nvidia langchain langchain-nvidia-ai-endpoints -e "../nemo_retriever[local]"

Configure the sample PDF, the local LanceDB path, and the VL embedding model. The ingest and query paths must use the same embedding model.

In [ ]:
from pathlib import Path

from nemo_retriever import create_ingestor
from nemo_retriever.model import VL_EMBED_MODEL
from nemo_retriever.retriever import Retriever

pdf_path = Path("../data/multimodal_test.pdf").resolve()
LANCEDB_URI = str(Path("./llama_index_multimodal_lancedb").resolve())
TABLE_NAME = "nemo-retriever"
TOP_K = 5

VL_EMBED_MODEL

Run NeMo Retriever in-process to extract multimodal content, create VL embeddings, and upload the embedded chunks to local LanceDB.

In [ ]:
ingestor = (
    create_ingestor(run_mode="inprocess")
    .files([str(pdf_path)])
    .extract()
    .embed(
        model_name=VL_EMBED_MODEL,
        embed_model_name=VL_EMBED_MODEL,
        embed_granularity="page",
        local_ingest_embed_backend="hf",
    )
    .vdb_upload(
        vdb_op="lancedb",
        vdb_kwargs={"uri": LANCEDB_URI, "table_name": TABLE_NAME, "overwrite": True},
    )
)

result_df = ingestor.ingest()
result_df[["text"]].head()

Create a small LlamaIndex retriever adapter around `nemo_retriever.retriever.Retriever`. NeMo Retriever performs the query embedding and LanceDB vector search; LlamaIndex handles the query engine and response synthesis.

In [ ]:
import json
from typing import Any

from llama_index.core import QueryBundle
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore, TextNode
from llama_index.llms.nvidia import NVIDIA


def metadata_from_hit(hit: dict[str, Any]) -> dict[str, Any]:
    metadata = {k: v for k, v in hit.items() if k != "text"}
    for key in ("metadata", "source"):
        value = metadata.get(key)
        if isinstance(value, str):
            try:
                metadata[key] = json.loads(value)
            except json.JSONDecodeError:
                pass
    return metadata


def score_from_hit(hit: dict[str, Any]) -> float | None:
    try:
        return 1.0 / (1.0 + float(hit["_distance"]))
    except (KeyError, TypeError, ValueError):
        return None


class NemoRetrieverLlamaIndex(BaseRetriever):
    def __init__(self, retriever: Retriever, top_k: int = 5) -> None:
        self._retriever = retriever
        self._top_k = top_k
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> list[NodeWithScore]:
        hits = self._retriever.query(query_bundle.query_str, top_k=self._top_k)
        nodes = []
        for rank, hit in enumerate(hits, start=1):
            metadata = metadata_from_hit(hit)
            metadata["rank"] = rank
            node = TextNode(text=str(hit.get("text", "")), metadata=metadata)
            nodes.append(NodeWithScore(node=node, score=score_from_hit(hit)))
        return nodes


nemo_retriever = Retriever(
    top_k=TOP_K,
    embed_kwargs={
        "model_name": VL_EMBED_MODEL,
        "embed_model_name": VL_EMBED_MODEL,
        "local_ingest_embed_backend": "hf",
    },
    vdb_kwargs={"uri": LANCEDB_URI, "table_name": TABLE_NAME},
)
llama_retriever = NemoRetrieverLlamaIndex(nemo_retriever, top_k=TOP_K)

Use an NVIDIA-hosted chat model to synthesize an answer from the retrieved chunks. Set `NVIDIA_API_KEY` in the environment before running this cell.

In [ ]:
import os

if not os.environ.get("NVIDIA_API_KEY"):
    raise EnvironmentError("Set NVIDIA_API_KEY to use the NVIDIA-hosted LLM for answer generation.")

llm = NVIDIA(model="nvidia/nemotron-3-super-120b-a12b", is_chat_model=True)
response_synthesizer = get_response_synthesizer(llm=llm)
query_engine = RetrieverQueryEngine(
    retriever=llama_retriever,
    response_synthesizer=response_synthesizer,
)

response = query_engine.query("What is the dog doing and where?")
response.response

In [ ]:
# Expected answer: the dog is chasing a squirrel in the front yard.
# Exact wording may vary because the final response is generated by the configured LLM.